# W2 Day6 (04/19 周六): [实战] 合成数据 Pipeline v1 ★

## 这是 P1 阶段的锚点项目！

整个 W1-W3 的所有学习都是为了今天能从零搭出这个 Pipeline:
- W1 的 Python 工程基础 (并发、装饰器) → 数据 pipeline 的脚手架
- W2 的 Transformer/CLIP/Diffusion → Pipeline 的核心组件
- D6 (今天): 把它们串成端到端可复现的项目

## 学习目标
- **理论 (1-1.5h)**: 合成数据有效性 / domain gap; 本体结构化描述 → ControlNet 生成 → CLIP 筛选 → 混合训练的完整理论框架
- **实践 (5h)**: 端到端实现: 本体描述 → 生成 → 筛选 → ResNet/ViT 混合训练 → 对比 baseline
- **产出物**: 合成 Pipeline v1 (含消融实验骨架，下周 D3 完成 Docker 化和 GitHub 发布)

## 与博士论文的关系

你的论文方法 (AiC under revision) 已经做完了，**今天是把它工程化**:
- 论文: 一次性实验 + 几个 figure
- 今天: 可复现的 pipeline + CLI 接口 + 消融实验框架 + 评估自动化
- 面试时: 「我把博士论文的方法升级成了可复现的工程 pipeline，10 分钟可以跑一轮新的本体 → 生成 → 训练对比」

## 核心问题 (面试锚点)
1. **为什么用合成数据？** 真实标注昂贵 (施工场景特别难标)、长尾类别难收集
2. **domain gap 怎么解决？** ControlNet 注入真实结构 + 严筛 + 比例消融
3. **筛选策略消融实验怎么设计？** 阈值 (τ)、筛选信号 (CLIP/FID/uncertainty)、保留比例
4. **数据量扩大 10× 架构怎么改？** 流式生成、分布式 CLIP 推理、增量索引
5. **你这个 pipeline 跟随便一个 SDXL+ControlNet 脚本的差异？** 本体约束的结构化生成 + 系统化评估 + 可复现

---

## 目录
1. [Pipeline 总体设计](#1)
2. [Step 1: 本体 → Prompt 生成](#2)
3. [Step 2: 图像生成 (Mock SD/ControlNet)](#3)
4. [Step 3: CLIP Score 筛选](#4)
5. [Step 4: 混合数据集构建](#5)
6. [Step 5: ResNet vs ViT Backbone 训练](#6)
7. [Step 6: 实验结果对比](#7)
8. [消融实验框架](#8)
9. [Pipeline 工程化 (CLI / Config / Logger)](#9)
10. [总结 & GitHub 发布清单](#10)


---
## 1. Pipeline 总体设计 <a id='1'></a>

### 1.1 端到端流程

```
   ┌─────────────────────┐
   │ Ontology (OWL/JSON)  │  ← 你的 ConSE 本体
   │ (entities, props,    │
   │  relations)          │
   └──────────┬───────────┘
              │ 1. 结构化采样
              ↓
   ┌─────────────────────┐
   │ Prompt Generator     │  ← 模板化、可控
   │ "scaffolding +       │
   │ worker + helmet"     │
   └──────────┬───────────┘
              │ 2. 文本生图
              ↓
   ┌─────────────────────┐    ┌────────────────┐
   │ SD + ControlNet      │ ←  │ Condition Img  │  (canny/MLSD)
   │ (生成 N×k 张)         │    │ (来自真实样本) │
   └──────────┬───────────┘    └────────────────┘
              │ 3. CLIP 筛选
              ↓
   ┌─────────────────────┐
   │ CLIP Score Filter    │  保留 score > τ
   │ + (可选) 难度感知     │
   └──────────┬───────────┘
              │ 4. 混合
              ↓
   ┌─────────────────────┐
   │ Mixed Dataset        │  Real + Filtered Synthetic
   │ (with class labels)  │
   └──────────┬───────────┘
              │ 5. 训练
              ↓
   ┌──────────────────────────────────────┐
   │ Backbone Train (ResNet-50 / ViT-B)   │
   │ 对比 4 组实验:                       │
   │   A) 100% Real (baseline)            │
   │   B) 100% Synthetic                  │
   │   C) 50/50 Mixed                     │
   │   D) 30 Real + 70 Filtered Synthetic │
   └──────────────────────────────────────┘
              ↓
   ┌─────────────────────┐
   │ Eval: Acc, mAP,     │  在 Real Test Set 上
   │ per-class F1        │
   └─────────────────────┘
```

### 1.2 实战策略 (今天 vs 真实项目)

| 步骤 | 真实项目 (你将来要做) | 今天的演示 |
|---|---|---|
| 本体 | 完整 OWL (ConSE 几百实体) | 简化 JSON (5 类施工实体) |
| 生成 | SDXL + ControlNet, 数千张 | Mock 生成器 (合成几何图像) |
| CLIP 筛选 | 真 CLIP 模型 | Mock CLIP score |
| Backbone | ResNet-50 / ViT-B 完整训练 | TinyResNet / TinyViT |
| 评估 | 真实测试集 | 合成测试集 |

**为什么 mock？** 真实跑一遍 SDXL + 训练要 4-8 小时 + 多 GB 模型权重，不现实在 notebook 里完成。**Pipeline 的结构和接口跟真实一致**，明天换上真组件就能跑。

### 1.3 工程原则

1. **组件接口稳定**: 每一步都是 `input → output` 的纯函数 (除生成步)
2. **配置驱动**: 所有超参在一个 config dict / YAML
3. **可复现**: 全程 seed 固定
4. **日志完整**: 每一步的中间产物都保存


In [ ]:
import json
import os
import time
import math
import random
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 全局 seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 工作目录
WORK_DIR = Path('/tmp/synth_pipeline')
WORK_DIR.mkdir(exist_ok=True)
print(f"Device: {device}")
print(f"Work dir: {WORK_DIR}")


---
## 2. Step 1: 本体 → Prompt 生成 <a id='2'></a>

### 2.1 简化本体 (模拟你的 ConSE)

真实你的 ConSE 本体可能有几百个实体类、几十个属性、复杂的 relation。这里用 5 类做演示:

- `Entity (类别)`: scaffold / worker / crane / helmet / barrier
- `Attribute`: color, material, condition
- `Relation`: on, near, holding

### 2.2 为什么要本体？

直接 prompt SD 「a construction site」会产生**毫无结构的图像**。本体保证:
- **类别覆盖均衡**: 每个类有目标数量样本
- **属性组合可控**: 不会生成「红色木质钢筋」这种不存在的组合
- **关系约束**: scaffold 一定在 worker 下面，不会颠倒
- **可追踪标签**: 每张图都能精确知道它包含哪些实体 (用于自动标注)

这就是为什么你的论文叫 *Ontology-guided* 而不是 *prompt-engineered*。


In [ ]:
# 简化的施工场景本体 (JSON 版)
ONTOLOGY = {
    "classes": {
        "scaffold": {"materials": ["steel", "aluminum"], "conditions": ["new", "weathered"], "weight": 1.0},
        "worker":   {"materials": None, "conditions": ["working", "resting", "climbing"], "weight": 1.5},
        "crane":    {"materials": ["steel"], "conditions": ["operational"], "weight": 0.8},
        "helmet":   {"materials": ["plastic"], "conditions": ["red", "yellow", "white", "blue"], "weight": 1.2},
        "barrier":  {"materials": ["plastic", "metal"], "conditions": ["intact", "damaged"], "weight": 1.0},
    },
    "relations": [
        ("worker", "wearing", "helmet"),
        ("worker", "on", "scaffold"),
        ("worker", "near", "barrier"),
        ("crane", "above", "scaffold"),
    ],
    "scene_templates": [
        "a construction site with {subject}, {context}, daylight, photorealistic",
        "close-up view of {subject} {action}, {context}",
        "wide shot of {subject} on a construction site, {context}",
    ],
}

# 保存本体到磁盘 (工程实践: 数据资产可追溯)
with open(WORK_DIR / 'ontology.json', 'w') as f:
    json.dump(ONTOLOGY, f, indent=2)
print(f"本体保存到: {WORK_DIR / 'ontology.json'}")
print(f"  类别数: {len(ONTOLOGY['classes'])}")
print(f"  关系数: {len(ONTOLOGY['relations'])}")
print(f"  模板数: {len(ONTOLOGY['scene_templates'])}")


In [ ]:
@dataclass
class GenerationSpec:
    """单个生成请求的完整描述 (用于 prompt + 标签)"""
    target_class: str      # 主要目标类 (用于分类标签)
    prompt: str            # 完整 SD prompt
    negative_prompt: str   # 负面 prompt
    metadata: dict         # 本体追踪信息


def sample_from_ontology(target_class, n_samples, ontology=ONTOLOGY):
    """从本体采样生成规格"""
    cls_info = ontology["classes"][target_class]
    specs = []
    
    for _ in range(n_samples):
        material = random.choice(cls_info["materials"]) if cls_info["materials"] else None
        condition = random.choice(cls_info["conditions"]) if cls_info["conditions"] else None
        
        if material and condition:
            subject = f"a {condition} {material} {target_class}"
        elif condition:
            subject = f"a {condition} {target_class}"
        elif material:
            subject = f"a {material} {target_class}"
        else:
            subject = f"a {target_class}"
        
        related = [r for r in ontology["relations"] if r[0] == target_class or r[2] == target_class]
        if related and random.random() < 0.6:
            rel = random.choice(related)
            other = rel[2] if rel[0] == target_class else rel[0]
            context = f"{rel[1]} a {other}"
        else:
            context = "in a busy work site"
        
        action = random.choice(["working", "climbing", "signaling"])
        
        template = random.choice(ontology["scene_templates"])
        prompt = template.format(subject=subject, context=context, action=action)
        
        spec = GenerationSpec(
            target_class=target_class,
            prompt=prompt,
            negative_prompt="low quality, blurry, distorted, unrealistic",
            metadata={
                "material": material,
                "condition": condition,
                "context": context,
                "template_idx": ontology["scene_templates"].index(template),
            }
        )
        specs.append(spec)
    
    return specs


# 演示
specs = sample_from_ontology("worker", 3)
print("采样的生成规格:")
print("-" * 60)
for i, s in enumerate(specs):
    print(f"[{i}] target_class: {s.target_class}")
    print(f"    prompt:        {s.prompt}")
    print(f"    metadata:      {s.metadata}")
    print()


In [ ]:
# 全本体采样: 按权重为每个类生成 spec
def sample_all_classes(ontology, n_per_class=20):
    all_specs = []
    for cls in ontology["classes"]:
        n = int(n_per_class * ontology["classes"][cls]["weight"])
        specs = sample_from_ontology(cls, n, ontology)
        all_specs.extend(specs)
    return all_specs


N_PER_CLASS = 30
all_specs = sample_all_classes(ONTOLOGY, n_per_class=N_PER_CLASS)
print(f"总生成规格数: {len(all_specs)}")

from collections import Counter
cls_counts = Counter([s.target_class for s in all_specs])
print("\n类别分布:")
for c, n in cls_counts.items():
    print(f"  {c:10s}: {n}")

# 保存 specs
with open(WORK_DIR / 'specs.json', 'w') as f:
    json.dump([asdict(s) for s in all_specs], f, indent=2)
print(f"\nSpecs 保存到: {WORK_DIR / 'specs.json'}")


---
## 3. Step 2: 图像生成 (Mock SD/ControlNet) <a id='3'></a>

真实环境下这一步是:

```python
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-mlsd")
pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", 
                                                          controlnet=controlnet)
pipe.to("cuda")

for spec in specs:
    cond = load_mlsd_condition(spec.target_class)
    img = pipe(
        prompt=spec.prompt,
        negative_prompt=spec.negative_prompt,
        image=cond,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]
    img.save(...)
```

为了在 notebook 内能跑通 Pipeline 全流程，我们用 mock 生成器: **每个类生成有可识别模式的合成图像** (圆/方/三角等)。


In [ ]:
# Mock 图像生成器: 5 个类别 → 5 种几何图案 (with noise)
CLASS_TO_PATTERN = {
    "scaffold":  "vertical_lines",
    "worker":    "circle",
    "crane":     "l_shape",
    "helmet":    "semicircle",
    "barrier":   "horizontal_lines",
}

PATTERN_COLORS = {
    "vertical_lines":   (0.6, 0.6, 0.6),
    "circle":           (1.0, 0.7, 0.4),
    "l_shape":          (1.0, 1.0, 0.0),
    "semicircle":       (1.0, 0.0, 0.0),
    "horizontal_lines": (1.0, 0.5, 0.0),
}


def mock_generate(spec, size=32, seed=None):
    """模拟 SD/ControlNet 生成。真实场景下替换为 diffusion pipeline 调用"""
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random
    
    pattern = CLASS_TO_PATTERN[spec.target_class]
    color = np.array(PATTERN_COLORS[pattern])
    
    img = np.full((size, size, 3), 0.85)
    
    if pattern == "vertical_lines":
        for x in range(4, size - 4, 5):
            img[2:size-2, x:x+1] = color
        img[size//2-1:size//2+1, 4:size-4] = color * 0.7
    elif pattern == "circle":
        cx, cy = size // 2, size // 2
        r = 8 + rng.randint(-2, 3)
        for x in range(size):
            for y in range(size):
                if (x-cx)**2 + (y-cy)**2 <= r**2:
                    img[x, y] = color
    elif pattern == "l_shape":
        img[2:size-2, 2:5] = color
        img[2:5, 2:size-2] = color
    elif pattern == "semicircle":
        cx, cy = size // 2, size // 2 + 4
        r = 10
        for x in range(size):
            for y in range(size):
                if (x-cx)**2 + (y-cy)**2 <= r**2 and x <= cx:
                    img[x, y] = color
    elif pattern == "horizontal_lines":
        for x in range(6, size - 6, 4):
            img[x:x+2, 2:size-2] = color
    
    img += rng.randn(*img.shape) * 0.08
    img = np.clip(img, 0, 1)
    
    # 偶尔加严重失真 (模拟坏样本，让筛选有意义)
    if rng.random() < 0.15:
        if rng.random() < 0.5:
            img += rng.randn(*img.shape) * 0.4
            img = np.clip(img, 0, 1)
        else:
            shift = rng.randint(-5, 5)
            img = np.roll(img, shift, axis=0)
    
    return torch.from_numpy(img.transpose(2, 0, 1)).float()


# 生成所有规格的图像
print("开始生成 (mock SD + ControlNet)...")
t0 = time.time()
generated_images = []
for i, spec in enumerate(all_specs):
    img = mock_generate(spec, size=32, seed=i)
    generated_images.append((img, spec))
print(f"生成完成: {len(generated_images)} 张, {time.time()-t0:.2f}s")

# 可视化每个类别 4 个样本
fig, axes = plt.subplots(5, 4, figsize=(8, 10))
class_idx_map = {c: 0 for c in ONTOLOGY["classes"]}
for img, spec in generated_images:
    cls = spec.target_class
    if class_idx_map[cls] < 4:
        row = list(ONTOLOGY["classes"].keys()).index(cls)
        col = class_idx_map[cls]
        axes[row, col].imshow(img.permute(1, 2, 0).numpy())
        axes[row, col].set_title(f"{cls} #{class_idx_map[cls]}", fontsize=9)
        axes[row, col].axis('off')
        class_idx_map[cls] += 1
plt.suptitle('Mock 生成的合成数据 (5 类)', y=1.0)
plt.tight_layout()
plt.show()


---
## 4. Step 3: CLIP Score 筛选 <a id='4'></a>

### 4.1 筛选的两个目的

1. **过滤生成失败样本** (噪声大、走样、与 prompt 无关)
2. **难度分布控制**: 太简单的样本 (几乎是 prompt 直译) 训练价值低；太难的样本可能是 OOD

### 4.2 真实实现

```python
import clip
clip_model, preprocess = clip.load("ViT-B/32", device=device)

def clip_score(image, text):
    image_input = preprocess(image).unsqueeze(0).to(device)
    text_input = clip.tokenize([text]).to(device)
    
    with torch.no_grad():
        img_feat = clip_model.encode_image(image_input)
        txt_feat = clip_model.encode_text(text_input)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
    
    return (img_feat @ txt_feat.T).item() * 100
```

### 4.3 难度感知筛选 (你下周 D3 会做的进阶)

```
score 太低 → 过滤 (生成失败)
score 中等 → 保留 (中等难度，训练价值高)
score 太高 → 部分保留 (太简单，可能信息冗余)
```


In [ ]:
def mock_clip_score(img, prompt, true_class):
    """Mock CLIP score: 主要按图像质量(噪声水平)和类别一致性给分"""
    img_std = img.std().item()
    base_score = max(0, 30 - 80 * abs(img_std - 0.25))
    score = base_score + np.random.randn() * 3
    if true_class in prompt:
        score += 5
    return float(np.clip(score, 0, 50))


# 给每个生成样本算 score
print("计算 CLIP Score...")
scored = []
for img, spec in generated_images:
    s = mock_clip_score(img, spec.prompt, spec.target_class)
    scored.append((img, spec, s))

scores = np.array([s for _, _, s in scored])
print(f"Score 统计:")
print(f"  mean = {scores.mean():.2f}, std = {scores.std():.2f}")
print(f"  min = {scores.min():.2f}, max = {scores.max():.2f}")
print(f"  median = {np.median(scores):.2f}")

# 按类别分布
fig, ax = plt.subplots(figsize=(10, 4))
for cls in ONTOLOGY["classes"]:
    cls_scores = [s for _, sp, s in scored if sp.target_class == cls]
    ax.hist(cls_scores, bins=15, alpha=0.5, label=cls)
ax.set_xlabel('CLIP Score')
ax.set_ylabel('Count')
ax.set_title('CLIP Score 分布 (按类别)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 应用筛选: 保留前 70%
def filter_by_score(scored_items, keep_pct=70):
    threshold = np.percentile([s for _, _, s in scored_items], 100 - keep_pct)
    kept = [(img, sp, sc) for img, sp, sc in scored_items if sc >= threshold]
    return kept, threshold


KEEP_PCT = 70
filtered, threshold = filter_by_score(scored, keep_pct=KEEP_PCT)
print(f"筛选 (保留 top {KEEP_PCT}%):")
print(f"  阈值: {threshold:.2f}")
print(f"  保留: {len(filtered)} / {len(scored)}")


def show_samples(items, title, n=8):
    fig, axes = plt.subplots(1, n, figsize=(n * 1.5, 2))
    for i, (img, sp, sc) in enumerate(items[:n]):
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].set_title(f"{sp.target_class}\nscore={sc:.1f}", fontsize=8)
        axes[i].axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# 高分 (保留)
high_score = sorted(scored, key=lambda x: x[2], reverse=True)[:8]
show_samples(high_score, "Top 8 (高 CLIP Score, 保留)")

# 低分 (过滤掉)
low_score = sorted(scored, key=lambda x: x[2])[:8]
show_samples(low_score, "Bottom 8 (低 CLIP Score, 被过滤)")


---
## 5. Step 4: 混合数据集构建 <a id='5'></a>

### 5.1 四种实验组

| 组 | 真实数据 | 合成数据 | 目的 |
|---|---|---|---|
| A: Real Only | 100% | 0% | **baseline** |
| B: Synth Only | 0% | 100% | 极端: 纯合成能否 work |
| C: Mixed 50/50 | 50% | 50% | 经典混合 |
| D: Real + Filtered Synth | 30% | 70% (CLIP 筛选) | 你的论文方法 |

### 5.2 工程化数据集类

`MixedDataset` 同时支持加载真实/合成样本，标签和来源都可追踪。


In [ ]:
# 准备「真实」数据 (这里也是 mock，但每个类用纯净版本，不加严重噪声)
def make_real_dataset(n_per_class=40, size=32):
    real_items = []
    for cls in ONTOLOGY["classes"]:
        for i in range(n_per_class):
            spec = GenerationSpec(
                target_class=cls,
                prompt=f"a real {cls}",
                negative_prompt="",
                metadata={"source": "real"}
            )
            img = mock_generate(spec, size=size, seed=10000 + hash(cls) % 1000 + i)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            real_items.append((img, cls, "real"))
    return real_items


real_train = make_real_dataset(n_per_class=40)
real_test = make_real_dataset(n_per_class=15)
print(f"Real train: {len(real_train)}")
print(f"Real test:  {len(real_test)}")

synth_filtered = [(img, sp.target_class, "synthetic") for img, sp, sc in filtered]
print(f"Synth filtered: {len(synth_filtered)}")

CLASSES = sorted(ONTOLOGY["classes"].keys())
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
print(f"\nClasses ({NUM_CLASSES}): {CLASSES}")


In [ ]:
class MixedDataset(torch.utils.data.Dataset):
    """混合真实 + 合成数据，记录来源"""
    
    def __init__(self, items, augment=False):
        self.items = items
        self.augment = augment
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        img, cls_name, source = self.items[idx]
        label = CLS_TO_IDX[cls_name]
        
        if self.augment:
            if random.random() < 0.5:
                img = torch.flip(img, dims=[2])
            if random.random() < 0.5:
                img = img + torch.randn_like(img) * 0.02
                img = img.clamp(0, 1)
        
        return img, label, source


def build_experiment_dataset(real_items, synth_items, exp_name):
    if exp_name == 'A_real_only':
        return MixedDataset(real_items, augment=True)
    elif exp_name == 'B_synth_only':
        return MixedDataset(synth_items, augment=True)
    elif exp_name == 'C_mixed_50_50':
        n = min(len(real_items), len(synth_items))
        items = real_items[:n] + synth_items[:n]
        random.shuffle(items)
        return MixedDataset(items, augment=True)
    elif exp_name == 'D_real_30_synth_70':
        n_real = int(0.3 * (len(real_items) + len(synth_items)))
        n_synth = len(real_items) + len(synth_items) - n_real
        items = real_items[:min(n_real, len(real_items))] + \
                synth_items[:min(n_synth, len(synth_items))]
        random.shuffle(items)
        return MixedDataset(items, augment=True)
    else:
        raise ValueError(exp_name)


EXP_NAMES = ['A_real_only', 'B_synth_only', 'C_mixed_50_50', 'D_real_30_synth_70']
exp_datasets = {}
print("实验数据集大小:")
for n in EXP_NAMES:
    ds = build_experiment_dataset(real_train, synth_filtered, n)
    exp_datasets[n] = ds
    real_n = sum(1 for _, _, s in ds.items if s == 'real')
    synth_n = sum(1 for _, _, s in ds.items if s == 'synthetic')
    print(f"  {n:30s}: total={len(ds):3d}  real={real_n:3d}  synth={synth_n:3d}")

test_ds = MixedDataset(real_test, augment=False)
print(f"\n测试集 (real only): {len(test_ds)}")


---
## 6. Step 5: ResNet vs ViT Backbone 训练 <a id='6'></a>

### 6.1 两个 backbone

- **TinyResNet**: 3 个 ResBlock 级联 + GAP + Linear
- **TinyViT**: Patch embed (4×4) + 2 layer Transformer + CLS token

### 6.2 为什么对比 ResNet vs ViT 在合成数据上的表现？

研究发现 (你论文相关方向):
- ResNet 用 CNN 的归纳偏置 (locality, translation equivariance) → 在小数据集上更稳
- ViT 没有这些先验，**对数据量更敏感** → 合成数据补充时收益更大
- 这是「合成数据 + ViT」组合在论文中常见的原因


In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c),
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class TinyResNet(nn.Module):
    """3-stage tiny ResNet for 32×32 input"""
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.layer1 = ResBlock(32, 32, stride=1)
        self.layer2 = ResBlock(32, 64, stride=2)
        self.layer3 = ResBlock(64, 128, stride=2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)


class TinyViT(nn.Module):
    """Tiny ViT for 32×32 input, patch 4 → 64 patches"""
    def __init__(self, num_classes, patch_size=4, embed_dim=96, depth=2, heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        n_patches = (32 // patch_size) ** 2
        
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, heads, embed_dim*4, 
                                        dropout=0.1, activation='gelu', 
                                        batch_first=True, norm_first=True)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x[:, 0])
        return self.head(x)


m_res = TinyResNet(NUM_CLASSES)
m_vit = TinyViT(NUM_CLASSES)
print(f"TinyResNet 参数: {sum(p.numel() for p in m_res.parameters()):,}")
print(f"TinyViT    参数: {sum(p.numel() for p in m_vit.parameters()):,}")


In [ ]:
def train_one_run(model_cls, dataset, test_dataset, epochs=12, batch_size=32, lr=1e-3,
                   model_kwargs=None):
    model_kwargs = model_kwargs or {}
    model = model_cls(NUM_CLASSES, **model_kwargs).to(device)
    
    def collate(batch):
        imgs = torch.stack([b[0] for b in batch])
        labels = torch.tensor([b[1] for b in batch])
        sources = [b[2] for b in batch]
        return imgs, labels, sources
    
    train_loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_loss': [], 'test_acc': []}
    best_acc = 0
    
    for ep in range(epochs):
        model.train()
        ep_loss = 0; ep_total = 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * imgs.size(0)
            ep_total += imgs.size(0)
        scheduler.step()
        train_loss = ep_loss / ep_total
        
        model.eval()
        correct = 0; total = 0
        with torch.no_grad():
            for imgs, labels, _ in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                correct += (logits.argmax(-1) == labels).sum().item()
                total += imgs.size(0)
        test_acc = correct / total
        history['train_loss'].append(train_loss)
        history['test_acc'].append(test_acc)
        if test_acc > best_acc:
            best_acc = test_acc
    
    return best_acc, history


# 跑全部 4 组 × 2 backbone (mini benchmark)
EPOCHS = 8
results = {}
print(f"\n开始 4×2=8 组实验 (epochs={EPOCHS})...")
print("=" * 70)

for backbone_name, backbone_cls in [('ResNet', TinyResNet), ('ViT', TinyViT)]:
    for exp_name in EXP_NAMES:
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        t0 = time.time()
        ds = exp_datasets[exp_name]
        best_acc, hist = train_one_run(backbone_cls, ds, test_ds, epochs=EPOCHS)
        elapsed = time.time() - t0
        results[(backbone_name, exp_name)] = {'best_acc': best_acc, 'history': hist}
        print(f"  [{backbone_name:6s}] {exp_name:30s}: best_acc={best_acc:.3f} ({elapsed:.1f}s)")


---
## 7. Step 6: 实验结果对比 <a id='7'></a>


In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'A_real_only': '#4CAF50', 'B_synth_only': '#F44336', 
          'C_mixed_50_50': '#2196F3', 'D_real_30_synth_70': '#FF9800'}

for ax, backbone in zip(axes, ['ResNet', 'ViT']):
    for exp_name in EXP_NAMES:
        hist = results[(backbone, exp_name)]['history']
        ax.plot(hist['test_acc'], 'o-', label=exp_name.replace('_', ' '),
                color=colors[exp_name], linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Real Test Accuracy')
    ax.set_title(f'{backbone} on Real Test Set')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('合成数据 Pipeline: 4 组实验 × 2 backbone', y=1.02)
plt.tight_layout()
plt.show()

# 最终准确率柱状图
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(EXP_NAMES))
width = 0.35

resnet_accs = [results[('ResNet', n)]['best_acc'] for n in EXP_NAMES]
vit_accs = [results[('ViT', n)]['best_acc'] for n in EXP_NAMES]

bars1 = ax.bar(x - width/2, resnet_accs, width, label='ResNet', color='#1976D2')
bars2 = ax.bar(x + width/2, vit_accs, width, label='ViT', color='#E64A19')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([n.replace('_', '\n') for n in EXP_NAMES])
ax.set_ylabel('Best Test Accuracy on Real')
ax.set_title('合成数据 Pipeline 实验结果')
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 输出表格
print("\n" + "=" * 70)
print("最终结果表 (Real Test Acc)")
print("=" * 70)
print(f"{'Experiment':<28s} {'ResNet':>10s} {'ViT':>10s} {'Δ (ViT-Res)':>12s}")
print("-" * 70)
for n in EXP_NAMES:
    r = results[('ResNet', n)]['best_acc']
    v = results[('ViT', n)]['best_acc']
    print(f"{n:<28s} {r:>10.3f} {v:>10.3f} {v-r:>+12.3f}")
print("\n💡 解读 (典型预期):")
print("   - A (real only) 是 baseline")
print("   - B (synth only) 通常低于 A (domain gap)")
print("   - C (mixed 50/50) 应接近或略超 A")
print("   - D (real 30 + synth 70 筛选) 在小真实数据下应明显优于 A")
print("   - ViT 通常从合成数据中获益更多")


---
## 8. 消融实验框架 <a id='8'></a>

W3D3 你要完成「合成数据消融 + 终版」。今天先把消融实验的**接口和数据结构**搭好，下周直接填充结果。

### 8.1 消融维度

| 维度 | 取值 |
|---|---|
| **CLIP 阈值 τ** | {None, 20, 25, 30, 35} |
| **真/合成混合比** | {100/0, 70/30, 50/50, 30/70, 0/100} |
| **筛选信号** | {CLIP, FID, uncertainty (entropy)} |
| **Backbone** | {ResNet-50, ViT-B/16} |
| **数据增强** | {none, basic, mixup} |


In [ ]:
@dataclass
class AblationConfig:
    """单次消融实验的完整配置"""
    name: str
    keep_pct: float = 70.0
    real_synth_ratio: tuple = (50, 50)
    backbone: str = 'ResNet'
    epochs: int = 8
    seed: int = 42
    augment: bool = True


class AblationRunner:
    """消融实验运行器 (用于 D3 完整消融)"""
    
    def __init__(self, real_train, real_test, scored_synth):
        self.real_train = real_train
        self.real_test = real_test
        self.scored_synth = scored_synth
        self.results = []
    
    def run(self, config):
        torch.manual_seed(config.seed)
        np.random.seed(config.seed)
        random.seed(config.seed)
        
        if config.keep_pct >= 100:
            kept_synth = self.scored_synth
        else:
            kept, _ = filter_by_score(self.scored_synth, keep_pct=config.keep_pct)
            kept_synth = kept
        synth_items = [(img, sp.target_class, "synthetic") for img, sp, _ in kept_synth]
        
        r_pct, s_pct = config.real_synth_ratio
        total_n = len(self.real_train)
        n_real = int(total_n * r_pct / 100)
        n_synth = int(total_n * s_pct / 100)
        items = self.real_train[:n_real] + synth_items[:min(n_synth, len(synth_items))]
        random.shuffle(items)
        train_ds = MixedDataset(items, augment=config.augment)
        test_ds = MixedDataset(self.real_test, augment=False)
        
        backbone_cls = TinyResNet if config.backbone == 'ResNet' else TinyViT
        best_acc, hist = train_one_run(backbone_cls, train_ds, test_ds, epochs=config.epochs)
        
        result = {
            'config': asdict(config),
            'train_size': len(items),
            'real_n': n_real,
            'synth_n': min(n_synth, len(synth_items)),
            'best_acc': best_acc,
            'final_acc': hist['test_acc'][-1],
        }
        self.results.append(result)
        return result


# 演示: 运行 3 组消融
runner = AblationRunner(real_train, real_test, scored)

ablation_configs = [
    AblationConfig(name='abl_strict_filter',  keep_pct=50, real_synth_ratio=(30, 70), backbone='ResNet', epochs=6),
    AblationConfig(name='abl_loose_filter',   keep_pct=90, real_synth_ratio=(30, 70), backbone='ResNet', epochs=6),
    AblationConfig(name='abl_no_synth',       keep_pct=70, real_synth_ratio=(100, 0), backbone='ResNet', epochs=6),
]

print("运行消融实验 (mini)...")
print("=" * 70)
for cfg in ablation_configs:
    res = runner.run(cfg)
    print(f"  [{cfg.name:25s}] keep={cfg.keep_pct:>4.0f}% "
          f"ratio={cfg.real_synth_ratio} → best_acc={res['best_acc']:.3f}")

with open(WORK_DIR / 'ablation_results.json', 'w') as f:
    json.dump(runner.results, f, indent=2, default=str)
print(f"\n消融结果保存到: {WORK_DIR / 'ablation_results.json'}")
print("\n💡 D3 真实消融建议: 5×5 网格 (筛选阈值 × 混合比) + 2 backbone = 50 次实验")


---
## 9. Pipeline 工程化 <a id='9'></a>

### 9.1 项目结构

```
synthetic-pipeline/
├── README.md
├── requirements.txt
├── Dockerfile
├── configs/
│   ├── default.yaml
│   └── ablation/
│       └── exp_001.yaml
├── data/
│   ├── ontology/conse.json     # 你的 ConSE 本体
│   ├── conditions/             # ControlNet 条件图
│   └── real/                   # 真实样本
├── src/
│   ├── ontology.py             # 本体加载 + spec 采样 (今天 §2)
│   ├── generator.py            # SD/ControlNet 包装 (今天 §3)
│   ├── filter.py               # CLIP/FID/uncertainty 筛选 (今天 §4)
│   ├── dataset.py              # MixedDataset (今天 §5)
│   ├── models.py               # ResNet / ViT (今天 §6)
│   ├── train.py                # 训练循环 (今天 §6)
│   └── ablation.py             # AblationRunner (今天 §8)
├── scripts/
│   ├── generate.py             # CLI: 本体 → 生成
│   ├── filter.py               # CLI: 筛选
│   ├── train.py                # CLI: 训练
│   └── run_ablation.py         # CLI: 跑消融
├── notebooks/
│   ├── 01_pipeline_demo.ipynb  ← 今天的 notebook
│   └── 02_ablation_analysis.ipynb
└── results/
    ├── runs/{timestamp}/        # 每次实验的产物
    └── figures/                 # 论文级图表
```


In [ ]:
@dataclass
class PipelineConfig:
    """整条 pipeline 的配置"""
    ontology_path: str = str(WORK_DIR / 'ontology.json')
    output_dir: str = str(WORK_DIR / 'runs')
    n_per_class: int = 30
    image_size: int = 32
    sd_model: str = 'runwayml/stable-diffusion-v1-5'
    controlnet_model: str = 'lllyasviel/sd-controlnet-mlsd'
    num_inference_steps: int = 30
    guidance_scale: float = 7.5
    filter_type: str = 'clip'
    clip_threshold: float = 25.0
    keep_pct: Optional[float] = 70.0
    backbone: str = 'ResNet'
    real_synth_ratio: tuple = (30, 70)
    epochs: int = 12
    batch_size: int = 32
    lr: float = 1e-3
    seed: int = 42


cfg = PipelineConfig()
print("Pipeline Config:")
for k, v in asdict(cfg).items():
    print(f"  {k:25s} = {v}")

config_path = WORK_DIR / 'pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(asdict(cfg), f, indent=2)
print(f"\nConfig 保存到: {config_path}")


In [ ]:
# 简易日志器 (生产中用 wandb / mlflow)
class SimpleLogger:
    def __init__(self, run_name=None):
        if run_name is None:
            run_name = time.strftime('%Y%m%d_%H%M%S')
        self.run_dir = WORK_DIR / 'runs' / run_name
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.metrics = []
        self.start_time = time.time()
    
    def log(self, step, **kwargs):
        entry = {'step': step, 'time': time.time() - self.start_time, **kwargs}
        self.metrics.append(entry)
    
    def save(self):
        with open(self.run_dir / 'metrics.jsonl', 'w') as f:
            for m in self.metrics:
                f.write(json.dumps(m) + '\n')
        return self.run_dir


# 端到端 Pipeline 入口函数 (CLI 风格)
def run_pipeline(cfg, real_train, real_test):
    print("=" * 70)
    print(f"Pipeline 运行: backbone={cfg.backbone}, ratio={cfg.real_synth_ratio}")
    print("=" * 70)
    
    logger = SimpleLogger()
    
    # Step 1
    with open(cfg.ontology_path) as f:
        ont = json.load(f)
    specs = sample_all_classes(ont, n_per_class=cfg.n_per_class)
    logger.log(step=0, stage='spec', n=len(specs))
    print(f"[1/5] 采样 {len(specs)} 个生成规格")
    
    # Step 2
    gen_imgs = []
    for i, sp in enumerate(specs):
        gen_imgs.append((mock_generate(sp, size=cfg.image_size, seed=i), sp))
    logger.log(step=1, stage='generate', n=len(gen_imgs))
    print(f"[2/5] 生成 {len(gen_imgs)} 张图像")
    
    # Step 3
    scored_local = [(img, sp, mock_clip_score(img, sp.prompt, sp.target_class)) 
                    for img, sp in gen_imgs]
    if cfg.keep_pct is not None:
        kept, threshold = filter_by_score(scored_local, keep_pct=cfg.keep_pct)
    else:
        kept = [s for s in scored_local if s[2] >= cfg.clip_threshold]
        threshold = cfg.clip_threshold
    synth_items = [(img, sp.target_class, "synthetic") for img, sp, _ in kept]
    logger.log(step=2, stage='filter', n=len(kept), threshold=threshold)
    print(f"[3/5] CLIP 筛选: {len(kept)}/{len(scored_local)} 保留 (阈值 {threshold:.2f})")
    
    # Step 4
    r_pct, s_pct = cfg.real_synth_ratio
    n_real = int(len(real_train) * r_pct / 100)
    n_synth = int(len(real_train) * s_pct / 100)
    items = real_train[:n_real] + synth_items[:min(n_synth, len(synth_items))]
    random.shuffle(items)
    train_ds = MixedDataset(items, augment=True)
    test_ds = MixedDataset(real_test, augment=False)
    logger.log(step=3, stage='build_dataset', n=len(train_ds))
    print(f"[4/5] 混合数据集: {len(items)} samples (real={n_real}, synth={min(n_synth, len(synth_items))})")
    
    # Step 5
    print(f"[5/5] 训练 {cfg.backbone}...")
    backbone_cls = TinyResNet if cfg.backbone == 'ResNet' else TinyViT
    best_acc, hist = train_one_run(backbone_cls, train_ds, test_ds, 
                                     epochs=cfg.epochs, batch_size=cfg.batch_size, lr=cfg.lr)
    for ep, (l, a) in enumerate(zip(hist['train_loss'], hist['test_acc'])):
        logger.log(step=4, stage='train', epoch=ep, train_loss=l, test_acc=a)
    
    run_dir = logger.save()
    print(f"\n✅ 完成! best_test_acc = {best_acc:.4f}")
    print(f"✅ 日志: {run_dir}")
    
    return best_acc, hist, run_dir


# 跑一次端到端 demo
demo_cfg = PipelineConfig(epochs=6, n_per_class=20)
acc, hist, run_dir = run_pipeline(demo_cfg, real_train, real_test)


---
## 10. 总结 & GitHub 发布清单 <a id='10'></a>

### 今日成果

✅ **端到端 Pipeline 跑通** — 从本体到训练对比，可运行可复现  
✅ **5 个核心组件可独立替换** — `ontology / generator / filter / dataset / train`  
✅ **4×2 实验对比矩阵** — 4 数据组 × 2 backbone  
✅ **消融运行器骨架** — D3 直接填充 50 组实验  
✅ **配置驱动 + 日志** — 工程级别基础设施

### 真实环境替换清单 (D3 之前)

| Mock 组件 | 替换为 |
|---|---|
| `mock_generate` | `diffusers.StableDiffusionControlNetPipeline` |
| `mock_clip_score` | `clip.load("ViT-B/32") + cosine sim` |
| `make_real_dataset` | 真实施工数据集加载器 |
| `TinyResNet` / `TinyViT` | `torchvision.models.resnet50` / `timm.create_model('vit_base_patch16_224')` |

### 面试 talk track (这个项目讲 5 分钟)

> 「我把博士论文的合成数据方法升级成了端到端的可复现 pipeline。
> 
> **架构**: 5 个解耦组件 (ontology → generator → filter → dataset → train)，全部用配置驱动，每次 run 自动记录配置和指标。
> 
> **核心**: 用我自己的 ConSE 本体结构化生成 prompt，配合 ControlNet (MLSD 边缘) 控制建筑结构，CLIP Score 筛选低质量样本，最后混合训练。
> 
> **结果**: 真实数据 30% + 合成 70% (筛选后) 在 ViT-B 上比纯真实 baseline 高 X 个点，特别在长尾类别 (crane / specific helmet color) 上 mAP 提升明显。
> 
> **trade-off**: SDXL + ControlNet 推理慢 (~3s/图)，1 万张要 ~9 小时，所以做了批量化 + GPU 并发；CLIP 筛选阈值是消融出来的 (τ=25 是甜蜜点)；混合比 30:70 在我的数据规模下最好，更小数据集会更倾向 50:50。
> 
> **如果数据扩 10×**: pipeline 改流式，生成 → 即时筛选 → 增量索引到 FAISS，避免内存爆；FID 评估改流式增量算 (Wasserstein 估计)。
> 
> **GitHub**: 仓库已 Docker 化，README 有架构图和复现命令，单条命令 `python run_pipeline.py --config configs/default.yaml` 跑全流程。」

### GitHub 发布清单 (D3 完成时)

- [ ] `README.md`: 架构图 + 一键运行命令 + 实验结果表
- [ ] `requirements.txt`: pin 版本
- [ ] `Dockerfile`: 多阶段构建，最小镜像
- [ ] `docker-compose.yml`: 一键启动 (含 GPU 配置)
- [ ] `configs/default.yaml`: 默认配置
- [ ] `configs/ablation/*.yaml`: 消融配置
- [ ] `src/`: 源码 (今天的代码模块化)
- [ ] `scripts/run_pipeline.py`: 入口 CLI
- [ ] `notebooks/01_pipeline_demo.ipynb`: 今天这个
- [ ] `notebooks/02_ablation_analysis.ipynb`: D3 完成
- [ ] `tests/`: 至少每个组件 1 个 unit test
- [ ] `.github/workflows/ci.yml`: GitHub Actions (lint + test)
- [ ] `LICENSE`
- [ ] 可选: `assets/architecture.png`, `assets/results.png`

### 明天 (D7) 复盘 + 工程深挖②

- 画 Transformer 完整计算图 (D1-D4 知识整合)
- CLIP 对比学习为什么 work (D2 深入)
- 工程深挖: Ontology Prompting 接口设计 / prompt 版本管理 / 评估 pipeline / 工程化改进


In [ ]:
print("=" * 60)
print("W2 Day6 完成! 🎉 锚点项目 v1 跑通")
print("=" * 60)
print(f"""
今日成果:
  ✅ 端到端 Pipeline (本体 → 生成 → 筛选 → 混合 → 训练)
  ✅ 5 个解耦组件，配置驱动
  ✅ 4×2 实验对比矩阵跑完
  ✅ AblationRunner + SimpleLogger 骨架
  ✅ Mock 组件全部接口与真实工具一致

工作目录: {WORK_DIR}
  ├── ontology.json
  ├── specs.json
  ├── pipeline_config.json
  ├── ablation_results.json
  └── runs/{{timestamp}}/metrics.jsonl

下一步:
  → D7 复盘: 把 W2 全周知识 + 这个项目串成 talk track
  → W3 D3: 真组件替换 + 完整消融 (50 组) + Docker + GitHub
""")